# non_sport_crypto (main) daily data diagnostic and demo export

This notebook is the preparation layer for the supervisor-facing EDA notebook. It selects one frozen live candle run, builds a one-row-per-market lookup, extracts one row per market-day, checks basic integrity and coverage, and writes stable plain-CSV artifacts.

It writes the full snapshot under data/demo/ and a filtered research view under data/demo_filtered/:

- main_market_metadata.csv: one row per market_id with text, dates, filters, and coverage;
- main_daily_candles.csv.gz: one row per market-day with quantitative fields;
- export_summary.json: run provenance and quality results.

The filtered view keeps markets that pass both the volume and lifetime filters. The raw SQLite layer remains the source of truth for the complete payload.

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "db/kalshi_daily_probability_dataset.sqlite").exists()
    ),
    Path.cwd(),
)
DB_PATH = PROJECT_ROOT / "db" / "kalshi_daily_probability_dataset.sqlite"
DATA_DIR = PROJECT_ROOT / "data" / "demo"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SELECTION_ID = "20260817T124811Z-3b15c8a0-637b555156e3"
SELECTION_GROUP = "non_sport_crypto"
CANDLE_RUN_ID = None

METADATA_PATH = DATA_DIR / "main_market_metadata.csv"
CANDLES_PATH = DATA_DIR / "main_daily_candles.csv.gz"
SUMMARY_PATH = DATA_DIR / "export_summary.json"

METADATA_COLUMNS = [
    "market_id", "market_ticker", "event_ticker", "series_ticker",
    "market_question", "market_subtitle", "yes_subtitle", "no_subtitle",
    "market_rules", "event_question", "event_subtitle", "series_title",
    "series_category", "series_frequency", "market_status", "market_result",
    "open_time", "close_time", "passes_volume_filter",
    "passes_lifetime_filter", "passes_both_filters", "volume_fp",
    "lifetime_days", "history_start_ts", "history_end_ts",
    "expected_daily_rows", "received_daily_rows", "first_observation_ts",
    "last_observation_ts", "download_status", "actual_candles_exported",
]

CANDLE_COLUMNS = [
    "market_id", "market_ticker", "series_ticker", "end_period_ts",
    "date_utc", "price_open", "price_low", "price_high", "price_close",
    "price_mean", "price_previous", "yes_bid_close", "yes_ask_close",
    "volume", "open_interest",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)
print("DATA_DIR:", DATA_DIR)
print("SELECTION_ID:", SELECTION_ID)
print("SELECTION_GROUP:", SELECTION_GROUP)

PROJECT_ROOT: /Users/sneddy/research/pred_markets_clean/daily_export
DB_PATH: /Users/sneddy/research/pred_markets_clean/daily_export/db/kalshi_daily_probability_dataset.sqlite
DATA_DIR: /Users/sneddy/research/pred_markets_clean/daily_export/data/demo
SELECTION_ID: 20260817T124811Z-3b15c8a0-637b555156e3
SELECTION_GROUP: non_sport_crypto


In [9]:
def read_sql(query, params=()):
    read_uri = f"file:{DB_PATH.resolve()}?mode=ro"
    with sqlite3.connect(read_uri, uri=True, timeout=60) as conn:
        return pd.read_sql_query(query, conn, params=params)

def parse_json(value):
    try:
        return json.loads(value) if value else {}
    except (TypeError, json.JSONDecodeError):
        return {}

runs = read_sql(
    """
    SELECT run_id, started_at_utc, finished_at_utc, status, source_mode,
           config_json, stats_json, error_text
    FROM metadata_runs
    WHERE json_extract(config_json, '$.stage') = 'daily_candles'
      AND json_extract(config_json, '$.selection_id') = ?
      AND json_extract(config_json, '$.selection_group') = ?
    ORDER BY started_at_utc DESC
    """,
    (SELECTION_ID, SELECTION_GROUP),
)

if CANDLE_RUN_ID is None:
    if runs.empty:
        raise RuntimeError("No daily-candle run found for the selected group.")
    selected_run = runs.iloc[0]
else:
    selected = runs.loc[runs["run_id"].eq(CANDLE_RUN_ID)]
    if selected.empty:
        raise RuntimeError(f"CANDLE_RUN_ID not found: {CANDLE_RUN_ID}")
    selected_run = selected.iloc[0]

CANDLE_RUN_ID = selected_run["run_id"]
RUN_STATUS = selected_run["status"]
RUN_SOURCE_MODE = selected_run["source_mode"]
RUN_CONFIG = parse_json(selected_run["config_json"])
RUN_STATS = parse_json(selected_run["stats_json"])

display(runs[[
    "run_id", "status", "source_mode", "started_at_utc",
    "finished_at_utc", "error_text"
]].head(10))

,run_id,status,source_mode,started_at_utc,finished_at_utc,error_text
0,20260824T111012Z-465be964,success,live,2026-08-24T11:10:12Z,2026-08-24T11:12:53Z,None
1,20260824T110000Z-d8ccec16,partial,live,2026-08-24T11:00:00Z,2026-08-24T11:10:04Z,None
2,20260824T104623Z-38d6f222,running,live,2026-08-24T10:46:23Z,NaN,None
3,20260824T104547Z-0686d500,running,live,2026-08-24T10:45:47Z,NaN,None
4,20260824T103557Z-f71511d9,running,live,2026-08-24T10:35:57Z,NaN,None
5,20260819T220943Z-fa731a62,partial,live,2026-08-19T22:09:43Z,2026-08-19T22:13:33Z,None


## 1. Prepare the market-level lookup

We start with one row per market. This lookup keeps the question, event, and series context in one place, while also recording lifecycle dates, filter decisions, and returned-candle coverage. Keeping this information separate from the daily rows lets the EDA inspect text without repeating long descriptions on every observation.

In [10]:
market_metadata = read_sql(
    """
    SELECT
        mh.market_id,
        mh.market_ticker,
        mh.event_ticker,
        mh.series_ticker,
        m.title AS market_question,
        m.subtitle AS market_subtitle,
        m.yes_sub_title AS yes_subtitle,
        m.no_sub_title AS no_subtitle,
        m.rules_primary AS market_rules,
        e.title AS event_question,
        e.sub_title AS event_subtitle,
        s.title AS series_title,
        s.category AS series_category,
        s.frequency AS series_frequency,
        m.status AS market_status,
        m.result AS market_result,
        m.open_time,
        m.close_time,
        mh.passes_volume_filter,
        mh.passes_lifetime_filter,
        mh.passes_both_filters,
        mh.volume_fp,
        mh.lifetime_days,
        mh.history_start_ts,
        mh.history_end_ts,
        mh.expected_daily_rows,
        mh.received_daily_rows,
        mh.first_observation_ts,
        mh.last_observation_ts,
        mh.status AS download_status
    FROM market_history_manifest mh
    LEFT JOIN raw_markets m ON m.market_id = mh.market_id
    LEFT JOIN raw_events e ON e.event_ticker = mh.event_ticker
    LEFT JOIN raw_series s
      ON s.series_ticker = COALESCE(NULLIF(m.series_ticker, ''), e.series_ticker)
    WHERE mh.run_id = ? AND mh.selection_id = ? AND mh.selection_group = ?
    ORDER BY mh.market_ticker
    """,
    (CANDLE_RUN_ID, SELECTION_ID, SELECTION_GROUP),
)

if market_metadata.empty:
    raise RuntimeError("The selected candle run has no market metadata.")

if not market_metadata["market_id"].is_unique:
    raise ValueError("Market metadata must contain one row per market_id.")

for column in [
    "passes_volume_filter", "passes_lifetime_filter", "passes_both_filters",
    "volume_fp", "lifetime_days", "history_start_ts", "history_end_ts",
    "expected_daily_rows", "received_daily_rows",
    "first_observation_ts", "last_observation_ts",
]:
    market_metadata[column] = pd.to_numeric(
        market_metadata[column], errors="coerce"
    )

for column in ["open_time", "close_time"]:
    market_metadata[column] = pd.to_datetime(
        market_metadata[column], utc=True, errors="coerce"
    )

display(market_metadata[[
    "market_id", "market_ticker", "market_question",
    "event_question", "series_title", "open_time", "close_time"
]].head(10))

,market_id,market_ticker,market_question,event_question,series_title,open_time,close_time
0,kalshi:AMAZONFTC-29DEC31,AMAZONFTC-29DEC31,Will a court find that Amazon has illegally maintained a monopoly?,Courts consider Amazon a monopoly?,Courts consider Amazon a monopoly,2023-10-23 21:00:00+00:00,2030-01-01 18:58:00+00:00
1,kalshi:APPLEFOLD-26DEC31,APPLEFOLD-26DEC31,"Will Apple announce a foldable phone by Dec 31, 2026?",When will Apple announce foldable iPhone?,Apple reveals foldable iPhone,2024-07-24 14:00:00+00:00,2027-01-01 04:59:00+00:00
2,kalshi:APPLEUS-29DEC31,APPLEUS-29DEC31,DOJ wins their anti-trust case against Apple?,Courts consider Apple a monopoly?,Apple DOJ lawsuit,2024-03-26 12:00:00+00:00,2030-01-01 15:00:00+00:00
3,kalshi:BEYONCEGENRE-30-AFA,BEYONCEGENRE-30-AFA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
4,kalshi:BEYONCEGENRE-30-DEA,BEYONCEGENRE-30-DEA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
5,kalshi:BEYONCEGENRE-30-RA,BEYONCEGENRE-30-RA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
6,kalshi:BEYONCEGENRE-30-TAA,BEYONCEGENRE-30-TAA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
7,kalshi:BEYONCEGENRE-30-TCA,BEYONCEGENRE-30-TCA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
8,kalshi:BEYONCEGENRE-30-TGA,BEYONCEGENRE-30-TGA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00
9,kalshi:BEYONCEGENRE-30-THA,BEYONCEGENRE-30-THA,Where will the next Beyonce album chart?,What genre will Beyonce’s next album be?,Next Beyonce album chart,2024-05-08 16:00:00+00:00,2030-01-01 15:00:00+00:00


## 2. Extract native daily candles

Next we keep one row per market-day. These are the quantitative observations used to trace probabilities, quotes, trading activity, and open interest over time. The question text stays in the market-level lookup and can be joined by market_id when needed.

In [11]:
candles = read_sql(
    """
    SELECT
        c.market_id,
        c.market_ticker,
        c.series_ticker,
        c.end_period_ts,
        date(c.end_period_ts, 'unixepoch') AS date_utc,
        c.price_open,
        c.price_low,
        c.price_high,
        c.price_close,
        c.price_mean,
        c.price_previous,
        c.yes_bid_close,
        c.yes_ask_close,
        c.volume,
        c.open_interest
    FROM raw_daily_candles c
    JOIN market_history_manifest mh
      ON mh.market_id = c.market_id
     AND mh.run_id = ?
     AND mh.selection_id = ?
     AND mh.selection_group = ?
    WHERE c.source_mode = 'live' AND c.run_id = ?
    ORDER BY c.market_ticker, c.end_period_ts
    """,
    (CANDLE_RUN_ID, SELECTION_ID, SELECTION_GROUP, CANDLE_RUN_ID),
)

if not candles.empty:
    candles["end_period_ts"] = pd.to_numeric(
        candles["end_period_ts"], errors="coerce"
    )
    candles["date_utc"] = pd.to_datetime(
        candles["date_utc"], utc=True, errors="coerce"
    )
    for column in [
        "price_open", "price_low", "price_high", "price_close",
        "price_mean", "price_previous", "yes_bid_close", "yes_ask_close",
        "volume", "open_interest",
    ]:
        candles[column] = pd.to_numeric(candles[column], errors="coerce")

display(candles.head(10))

,market_id,market_ticker,series_ticker,end_period_ts,date_utc,price_open,price_low,price_high,price_close,price_mean,price_previous,yes_bid_close,yes_ask_close,volume,open_interest
0,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768539600,2026-01-16 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
1,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768626000,2026-01-17 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
2,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768712400,2026-01-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
3,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768798800,2026-01-19 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
4,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768885200,2026-01-20 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
5,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1768971600,2026-01-21 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
6,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1769058000,2026-01-22 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
7,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1769144400,2026-01-23 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
8,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1769230800,2026-01-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.85,0.95,0.0,0.0
9,kalshi:KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY-27JAN-JPOW,KXCONGRESSTESTIFY,1769317200,2026-01-25 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.87,0.95,0.0,0.0


In [12]:
# A retry run copies the parent manifest but writes refreshed candles under a new run ID.
# Include the full parent chain so the export keeps previously successful candles.
RETRY_PARENT_RUN_ID = RUN_CONFIG.get("retry_parent_run_id")
candles = read_sql(
    """
    WITH RECURSIVE source_runs(run_id) AS (
        SELECT ?
        UNION
        SELECT json_extract(m.config_json, '$.retry_parent_run_id')
        FROM metadata_runs m
        JOIN source_runs sr ON sr.run_id = m.run_id
        WHERE json_extract(m.config_json, '$.retry_parent_run_id') IS NOT NULL
    )
    SELECT
        c.market_id,
        c.market_ticker,
        c.series_ticker,
        c.end_period_ts,
        date(c.end_period_ts, 'unixepoch') AS date_utc,
        c.price_open,
        c.price_low,
        c.price_high,
        c.price_close,
        c.price_mean,
        c.price_previous,
        c.yes_bid_close,
        c.yes_ask_close,
        c.volume,
        c.open_interest
    FROM raw_daily_candles c
    JOIN market_history_manifest mh
      ON mh.market_id = c.market_id
     AND mh.run_id = ?
     AND mh.selection_id = ?
     AND mh.selection_group = ?
    WHERE c.source_mode = 'live'
      AND c.run_id IN (SELECT run_id FROM source_runs)
    ORDER BY c.market_ticker, c.end_period_ts
    """,
    (CANDLE_RUN_ID, CANDLE_RUN_ID, SELECTION_ID, SELECTION_GROUP),
)

if not candles.empty:
    candles["end_period_ts"] = pd.to_numeric(
        candles["end_period_ts"], errors="coerce"
    )
    candles["date_utc"] = pd.to_datetime(
        candles["date_utc"], utc=True, errors="coerce"
    )
    for column in [
        "price_open", "price_low", "price_high", "price_close",
        "price_mean", "price_previous", "yes_bid_close", "yes_ask_close",
        "volume", "open_interest",
    ]:
        candles[column] = pd.to_numeric(candles[column], errors="coerce")

## 3. Validate the export

These checks tell us whether the exported keys, market-day rows, probability fields, volume, and coverage are internally consistent. A live run may be partial because API requests can fail, so that status stays visible rather than being hidden.

In [13]:
actual_counts = candles.groupby("market_id").size()
market_metadata["actual_candles_exported"] = (
    market_metadata["market_id"]
    .map(actual_counts)
    .fillna(0)
    .astype("int64")
)

if len(candles):
    first_date = str(candles["date_utc"].min())
    last_date = str(candles["date_utc"].max())
else:
    first_date = None
    last_date = None

price_close = candles["price_close"].dropna()
bid_close = candles["yes_bid_close"].dropna()
ask_close = candles["yes_ask_close"].dropna()

quality = {
    "duplicate_market_day_rows": int(
        candles.duplicated(["market_id", "end_period_ts"]).sum()
    ),
    "missing_candle_market_id_rows": int(candles["market_id"].isna().sum()),
    "candle_markets_missing_metadata": int(
        (~candles["market_id"].isin(market_metadata["market_id"])).sum()
    ),
    "price_close_below_zero_rows": int(price_close.lt(0).sum()),
    "price_close_above_one_rows": int(price_close.gt(1).sum()),
    "bid_close_below_zero_rows": int(bid_close.lt(0).sum()),
    "bid_close_above_one_rows": int(bid_close.gt(1).sum()),
    "ask_close_below_zero_rows": int(ask_close.lt(0).sum()),
    "ask_close_above_one_rows": int(ask_close.gt(1).sum()),
    "negative_volume_rows": int(candles["volume"].lt(0).sum()),
    "negative_open_interest_rows": int(candles["open_interest"].lt(0).sum()),
    "market_metadata_rows": int(len(market_metadata)),
    "exported_markets": int(candles["market_id"].nunique()),
    "market_rows_with_zero_exported_candles": int(
        market_metadata["actual_candles_exported"].eq(0).sum()
    ),
}

filter_counts = (
    market_metadata.groupby(
        ["passes_volume_filter", "passes_lifetime_filter", "passes_both_filters"],
        dropna=False,
    )
    .size()
    .reset_index(name="markets")
)

summary = {
    "export_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_id": SELECTION_ID,
    "selection_group": SELECTION_GROUP,
    "source_mode": RUN_SOURCE_MODE,
    "candle_run_id": CANDLE_RUN_ID,
    "retry_parent_run_id": RETRY_PARENT_RUN_ID,
    "run_status": RUN_STATUS,
    "run_started_at_utc": selected_run["started_at_utc"],
    "run_finished_at_utc": selected_run["finished_at_utc"],
    "market_metadata_rows": int(len(market_metadata)),
    "candle_rows": int(len(candles)),
    "markets_with_candles": int(candles["market_id"].nunique()),
    "first_candle_date_utc": first_date,
    "last_candle_date_utc": last_date,
    "filter_counts": filter_counts.to_dict(orient="records"),
    "quality": quality,
    "run_stats": RUN_STATS,
}

display(pd.DataFrame([quality]))

,duplicate_market_day_rows,missing_candle_market_id_rows,candle_markets_missing_metadata,price_close_below_zero_rows,price_close_above_one_rows,bid_close_below_zero_rows,bid_close_above_one_rows,ask_close_below_zero_rows,ask_close_above_one_rows,negative_volume_rows,negative_open_interest_rows,market_metadata_rows,exported_markets,market_rows_with_zero_exported_candles
0,0,0,0,0,0,0,0,0,0,0,0,47700,45102,2598


## 4. Write the plain CSV artifacts

The full CSV snapshot is the broad handoff for inspection. We also write a filtered view containing only markets that pass both selection criteria, so downstream analysis can compare the broad universe with a more focused research cohort without another API download.

In [14]:
metadata_output = market_metadata.copy()

for column in ["open_time", "close_time"]:
    metadata_output[column] = metadata_output[column].dt.strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )

metadata_output = metadata_output[METADATA_COLUMNS]
candles_output = candles[CANDLE_COLUMNS]

metadata_output.to_csv(METADATA_PATH, index=False)
candles_output.to_csv(CANDLES_PATH, index=False, compression="gzip")

with SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

display(pd.DataFrame([
    {"file": str(METADATA_PATH), "rows": len(metadata_output)},
    {"file": str(CANDLES_PATH), "rows": len(candles_output)},
    {"file": str(SUMMARY_PATH), "rows": 1},
]))

,file,rows
0,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo/main_market_metadata.csv,47700
1,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo/main_daily_candles.csv.gz,2562061
2,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo/export_summary.json,1


In [15]:
FILTERED_DATA_DIR = PROJECT_ROOT / "data" / "demo_filtered"
FILTERED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FILTERED_METADATA_PATH = FILTERED_DATA_DIR / "main_market_metadata.csv"
FILTERED_CANDLES_PATH = FILTERED_DATA_DIR / "main_daily_candles.csv.gz"
FILTERED_SUMMARY_PATH = FILTERED_DATA_DIR / "export_summary.json"
FILTER_MODE = "both"

filter_mask = market_metadata["passes_both_filters"].eq(1)
filtered_market_ids = market_metadata.loc[filter_mask, "market_id"]
filtered_metadata_output = metadata_output[
    metadata_output["market_id"].isin(filtered_market_ids)
].copy()
filtered_candles_output = candles_output[
    candles_output["market_id"].isin(filtered_market_ids)
].copy()

filtered_summary = dict(summary)
filtered_summary.update({
    "source_export": str(DATA_DIR),
    "source_filter_counts": summary["filter_counts"],
    "filter_mode": FILTER_MODE,
    "filter_definition": "passes_both_filters == 1",
    "filter_counts": [{
        "passes_volume_filter": 1,
        "passes_lifetime_filter": 1,
        "passes_both_filters": 1,
        "markets": int(len(filtered_metadata_output)),
    }],
    "min_volume_fp": RUN_CONFIG.get("min_volume_fp"),
    "min_lifetime_days": RUN_CONFIG.get("min_lifetime_days"),
    "market_metadata_rows": int(len(filtered_metadata_output)),
    "candle_rows": int(len(filtered_candles_output)),
    "markets_with_candles": int(
        filtered_candles_output["market_id"].nunique()
    ),
})
filtered_summary["quality"] = {
    "metadata_market_id_unique": bool(
        filtered_metadata_output["market_id"].is_unique
    ),
    "duplicate_market_day_rows": int(
        filtered_candles_output.duplicated(
            ["market_id", "end_period_ts"]
        ).sum()
    ),
    "candle_markets_missing_metadata": int(
        (~filtered_candles_output["market_id"].isin(
            filtered_metadata_output["market_id"]
        )).sum()
    ),
}

filtered_metadata_output.to_csv(FILTERED_METADATA_PATH, index=False)
filtered_candles_output.to_csv(FILTERED_CANDLES_PATH, index=False, compression="gzip")

with FILTERED_SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(filtered_summary, handle, indent=2, default=str)
display(pd.DataFrame([
    {"file": str(FILTERED_METADATA_PATH), "rows": len(filtered_metadata_output)},
    {"file": str(FILTERED_CANDLES_PATH), "rows": len(filtered_candles_output)},
    {"file": str(FILTERED_SUMMARY_PATH), "rows": 1},
]))

,file,rows
0,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo_filtered/main_market_metadata.csv,7545
1,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo_filtered/main_daily_candles.csv.gz,860012
2,/Users/sneddy/research/pred_markets_clean/daily_export/data/demo_filtered/export_summary.json,1


## Handoff to daily_data_eda.ipynb

The EDA notebook reads the full snapshot from data/demo/ and the filtered view from data/demo_filtered/.